# Adaptive baseline mortality

Native analysis for Figure 3 and Supplementary Figure 9. The registered residual subtracts the model's weather and infection contributions. Weekly fitted baseline rates are averaged across the ten native checkpoints.

[Original analysis notebook](https://github.com/ClimSocAna/sari-weather-mortality/blob/4402a80d3e490ad7964a3be12597043761bebda1/notebooks/03_adaptive_baseline_mortality.ipynb).

In [ ]:
from pathlib import Path
import sys

repo = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "configs" / "training.example.yaml").is_file())
sys.path.insert(0, str(repo))

import matplotlib.pyplot as plt
from notebooks.utils import AGE_LABELS, daily_index, ensemble, weekly_sum

from notebooks.utils import ANALYSIS_PERIOD, DEVICE, checkpoint_paths
from sari_t.inference import InferenceModel
from sari_t.run.checkpoint import load_checkpoint, load_run_config, restore_predictor


## Load models for inspection

Run the next cell to load member `000` for each setup into memory. Inspect `model` for the hybrid predictor or `models[setup]` for another predictor. Set `inspection_member` to a value from 0 through 9 to inspect another member. The analysis cells evaluate all ten members in a separate inference workspace.

In [ ]:
inspection_member = 0
setups = ['default']
checkpoints = {setup: checkpoint_paths(setup) for setup in setups}
models = {
    setup: restore_predictor(load_checkpoint(paths[inspection_member]), device="cpu").eval()
    for setup, paths in checkpoints.items()
}
model = models["default"]
model

## Reconstruct the registered baseline

In [ ]:
def summarize(model, result):
    prediction = result.predictions
    additional = prediction.total_deaths - result.scenarios["reference"]
    infection = prediction.sari_deaths + prediction.covid_deaths
    observed = model.labels.weekly_de_age_sex
    residual = observed - weekly_sum((additional + infection).sum(dim=1))
    rate = weekly_sum(model.baseline_rate.squeeze(1)) / 7
    return {"Observed": observed, "Residual": residual, "Rate": rate}

paths = checkpoints["default"]
inputs = load_run_config(paths[0].parents[1] / "config.yaml").data_config
inference_model = InferenceModel.from_checkpoint(
    paths[0], data_config=inputs, period=ANALYSIS_PERIOD, device=DEVICE,
)
estimates = ensemble(inference_model, paths, summarize)
weeks = daily_index()[::7]

## Figure 3 and Supplementary Figure 9 · Age-specific trajectories

In [ ]:
def plot_baseline(age_indices):
    fig, axes = plt.subplots(len(age_indices), 2, figsize=(14, 2.6 * len(age_indices)),
                             sharex=True, squeeze=False)
    for age, (deaths, rate) in zip(age_indices, axes):
        for sex, label in enumerate(("Men", "Women")):
            deaths.plot(weeks, estimates["Observed"][:, age, sex],
                        alpha=0.45, label=f"{label}, registered")
            deaths.plot(weeks, estimates["Residual"][:, age, sex],
                        label=f"{label}, baseline residual")
            rate.plot(weeks, estimates["Rate"][:, age, sex] * 100_000, label=label)
        deaths.set(ylabel=f"{AGE_LABELS[age]} · weekly deaths")
        rate.set(ylabel="Daily rate per 100,000")
        deaths.legend(frameon=False, ncol=2, fontsize=8)
        rate.legend(frameon=False, fontsize=8)
    fig.tight_layout()

plot_baseline([7, 9, 11])
plot_baseline(list(range(len(AGE_LABELS))))